# DQI Reed–Solomon / Berlekamp–Massey reproduction

This notebook reproduces the published `dqi-reed-solomon` discovered circuit and evaluator from [`andigu/uncompute`](https://github.com/andigu/uncompute), pinned to commit `64b5a13aa249f1360846657392e6c66817b26d0f`.

This is a **CPU-only baseline reproduction**, not a new optimization. The upstream task uses 70 syndrome symbols over GF(2^12) and evaluates the reversible decoder against its stated input distribution.

Expected resource metrics at the pinned revision: **442,397 Toffoli gates**, **1,837 peak live qubits**, **TQ-cost 812,683,289**. The evaluator also checks phase, syndrome preservation, and workspace cleanup.

The notebook saves a JSON summary, evaluator logs, and a ZIP bundle under `/kaggle/working`. GitHub Actions downloads these outputs from Kaggle and uploads them as a downloadable workflow artifact.

The upstream repository is licensed Apache-2.0; this notebook fetches the source at a pinned commit instead of copying its implementation.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version)
print('Running in CPU-only mode; no GPU is requested.')

In [ ]:
UPSTREAM_URL = 'https://github.com/andigu/uncompute.git'
UPSTREAM_COMMIT = '64b5a13aa249f1360846657392e6c66817b26d0f'
UPSTREAM = WORK / 'uncompute-upstream'

if UPSTREAM.exists():
    shutil.rmtree(UPSTREAM)
subprocess.run(['git', 'init', str(UPSTREAM)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'remote', 'add', 'origin', UPSTREAM_URL], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'fetch', '--depth=1', 'origin', UPSTREAM_COMMIT], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(UPSTREAM), 'rev-parse', 'HEAD'], text=True).strip()
print('Pinned upstream commit:', actual_commit)
assert actual_commit == UPSTREAM_COMMIT, (actual_commit, UPSTREAM_COMMIT)
assert (UPSTREAM / 'tasks/dqi-reed-solomon/circuits/discovered/circuit.py').is_file()
assert (UPSTREAM / 'tools/eval.py').is_file()
print('Located the BM circuit and official evaluator.')

In [ ]:
# Install the upstream project's locked Python environment and optional native evaluator kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.12'], check=True)
subprocess.run(['uv', 'sync', '--locked'], cwd=UPSTREAM, check=True)
# The native C kernel accelerates evaluation; the pure-Python evaluator remains the reference path.
subprocess.run(['make', 'kernel'], cwd=UPSTREAM, check=True)
print('Environment ready.')

In [ ]:
RESULTS = WORK / 'bm-reproduction-results'
RESULTS.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env['UNCOMPUTE_KERNEL'] = '1'
cmd = [
    'uv', 'run', 'python', 'tools/eval.py', 'dqi-reed-solomon',
    '--json', '--out', str(RESULTS), '--jobs', '4', '--timeout', '3600'
]
print('Running:', ' '.join(cmd))
run = subprocess.run(cmd, cwd=UPSTREAM, env=env, text=True, capture_output=True)
print('Exit code:', run.returncode)

# Persist logs immediately, even if the evaluator fails; these are downloaded by GitHub Actions.
stdout_path = WORK / 'evaluator-stdout.txt'
stderr_path = WORK / 'evaluator-stderr.txt'
stdout_path.write_text(run.stdout or '', encoding='utf-8')
stderr_path.write_text(run.stderr or '', encoding='utf-8')
(RESULTS / 'evaluator-stdout.txt').write_text(run.stdout or '', encoding='utf-8')
(RESULTS / 'evaluator-stderr.txt').write_text(run.stderr or '', encoding='utf-8')
if run.stdout:
    print('STDOUT (tail):\\n', run.stdout[-6000:])
if run.stderr:
    print('STDERR (tail):\\n', run.stderr[-6000:])

report_path = RESULTS / 'report.json'
report = None
if report_path.exists():
    try:
        report = json.loads(report_path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        report = None
if report is None:
    for line in reversed((run.stdout or '').splitlines()):
        try:
            candidate = json.loads(line)
            if isinstance(candidate, dict):
                report = candidate
                break
        except json.JSONDecodeError:
            continue
if report is None:
    report = {
        'verdict': 'NO_REPORT',
        'note': 'No JSON report could be read; inspect evaluator-stdout.txt and evaluator-stderr.txt.',
    }
if not report_path.exists():
    report_path.write_text(json.dumps(report, indent=2) + '\\n', encoding='utf-8')

metrics = report.get('metrics', {}) if isinstance(report, dict) else {}
summary = {
    'upstream_url': UPSTREAM_URL,
    'upstream_commit': UPSTREAM_COMMIT,
    'task': 'dqi-reed-solomon',
    'evaluator_exit_code': run.returncode,
    'verdict': report.get('verdict', report.get('status')),
    'toffoli': metrics.get('toffoli'),
    'peak_qubits': metrics.get('qubits'),
    'tq_cost': metrics.get('tq_cost'),
    'metrics': metrics,
}
summary_text = json.dumps(summary, indent=2) + '\\n'
(RESULTS / 'summary.json').write_text(summary_text, encoding='utf-8')
(WORK / 'summary.json').write_text(summary_text, encoding='utf-8')
(WORK / 'bm-reproduction-report.json').write_text(json.dumps(report, indent=2) + '\\n', encoding='utf-8')
manifest = {
    'task': 'dqi-reed-solomon',
    'upstream_url': UPSTREAM_URL,
    'upstream_commit': UPSTREAM_COMMIT,
    'python_version': sys.version,
    'evaluator_command': cmd,
    'evaluator_exit_code': run.returncode,
}
(RESULTS / 'run-manifest.json').write_text(json.dumps(manifest, indent=2) + '\\n', encoding='utf-8')
(WORK / 'run-manifest.json').write_text(json.dumps(manifest, indent=2) + '\\n', encoding='utf-8')

# A single downloadable bundle is convenient even when the evaluator exits unsuccessfully.
bundle_path = WORK / 'bm-reproduction-results.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(WORK))

print(json.dumps(summary, indent=2))
expected = {'toffoli': 442397, 'peak_qubits': 1837, 'tq_cost': 812683289}
failures = []
if run.returncode != 0:
    failures.append(f'evaluator exit code was {run.returncode}')
if summary['verdict'] != 'OK':
    failures.append(f"unexpected evaluator verdict: {summary['verdict']}")
for key, value in expected.items():
    if summary.get(key) != value:
        failures.append(f'{key}: expected {value}, got {summary.get(key)}')
if failures:
    raise RuntimeError('BM reproduction did not match baseline: ' + '; '.join(failures) + '. Logs and partial results were saved first.')
print('PASS: reproduced the published BM circuit resource metrics.')